# EE200 Summer 2025 - Signal Processing Project
# Image Transforms, Audio Analysis, and Frequency Domain Processing

This notebook covers:
1. Basic Image Operations (resize, crop, rotate)
2. 2D Discrete Fourier Transform (DFT)
3. Frequency Domain Filtering (LPF/HPF)
4. Audio Loading and Waveform Visualization
5. Time-Frequency Analysis (Spectrogram)

In [ ]:
# Cell 1: Setup and Imports
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import librosa
import librosa.display
from scipy.fft import fft2, ifft2, fftshift
import os

# Set up inline plotting
%matplotlib inline
plt.style.use('default')

# Get current directory for file paths
base_path = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
print(f"Working directory: {base_path}")

---
## Part A: Image Processing

In [ ]:
# Cell 2: Load and Display Original Images
# Load grayscale images
cat_img = Image.open('cat_gray.jpg')
dog_img = Image.open('dog_gray.jpg')

# Convert to numpy arrays for processing
cat_array = np.array(cat_img)
dog_array = np.array(dog_img)

print(f"Cat image shape: {cat_array.shape}")
print(f"Dog image shape: {dog_array.shape}")
print(f"Cat dtype: {cat_array.dtype}")
print(f"Dog dtype: {dog_array.dtype}")

# Display original images
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cat_img, cmap='gray')
axes[0].set_title('Cat (Grayscale)')
axes[0].axis('off')

axes[1].imshow(dog_img, cmap='gray')
axes[1].set_title('Dog (Grayscale)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 3: Basic Image Operations - Resize, Crop, Rotate
# Resize both images to 200x200
cat_resized = cat_img.resize((200, 200))
dog_resized = dog_img.resize((200, 200))

# Crop: (left, upper, right, lower) coordinates
cat_cropped = cat_img.crop((50, 50, 200, 200))
dog_cropped = dog_img.crop((50, 50, 200, 200))

# Rotate by 45 degrees counter-clockwise
cat_rotated = cat_img.rotate(45)
dog_rotated = dog_img.rotate(45)

# Display all operations for cat image
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(cat_img, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(cat_resized, cmap='gray')
axes[1].set_title('Resized (200x200)')
axes[1].axis('off')

axes[2].imshow(cat_cropped, cmap='gray')
axes[2].set_title('Cropped (50-200)')
axes[2].axis('off')

axes[3].imshow(cat_rotated, cmap='gray')
axes[3].set_title('Rotated (45°)')
axes[3].axis('off')

plt.tight_layout()
plt.show()

print("\nCat image operations completed!")

In [ ]:
# Cell 4: 2D Discrete Fourier Transform (DFT)
# Compute 2D FFT for cat image
cat_fft = fft2(cat_array)
dog_fft = fft2(dog_array)

# Shift zero frequency to center
cat_fft_shift = fftshift(cat_fft)
dog_fft_shift = fftshift(dog_fft)

# Compute magnitude spectrum (log scale for better visualization)
cat_magnitude = np.log(1 + np.abs(cat_fft_shift))
dog_magnitude = np.log(1 + np.abs(dog_fft_shift))

# Compute phase spectrum
cat_phase = np.angle(cat_fft_shift)
dog_phase = np.angle(dog_fft_shift)

print(f"FFT shape: {cat_fft.shape}")
print(f"Max magnitude (log): {cat_magnitude.max():.2f}")
print(f"Phase range: [{cat_phase.min():.2f}, {cat_phase.max():.2f}] radians")

# Display magnitude spectra
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(cat_magnitude, cmap='gray')
axes[0].set_title('Cat - Magnitude Spectrum (Log)')
axes[0].axis('off')

axes[1].imshow(dog_magnitude, cmap='gray')
axes[1].set_title('Dog - Magnitude Spectrum (Log)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Display Phase Spectra
# Display phase spectra
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im1 = axes[0].imshow(cat_phase, cmap='twilight', aspect='auto')
axes[0].set_title('Cat - Phase Spectrum')
axes[0].axis('off')
plt.colorbar(im1, ax=axes[0], fraction=0.046)

im2 = axes[1].imshow(dog_phase, cmap='twilight', aspect='auto')
axes[1].set_title('Dog - Phase Spectrum')
axes[1].axis('off')
plt.colorbar(im2, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()

print("Phase spectra show the phase angle of each frequency component.")
print("The central area contains the DC component and low frequencies.")

In [ ]:
# Cell 6: Frequency Domain Filtering - Create Filter Masks with Multiple Cutoffs
def create_ideal_filter(shape, cutoff, filter_type='low'):
    """
    Create an ideal filter in frequency domain.

    Parameters:
    - shape: tuple (M, N) image dimensions
    - cutoff: cutoff frequency D0
    - filter_type: 'low' for LPF, 'high' for HPF

    Returns:
    - filter mask of same shape
    """
    M, N = shape
    # Create coordinate grids centered at zero
    u = np.arange(-M//2, M//2)
    v = np.arange(-N//2, N//2)
    V, U = np.meshgrid(v, u)

    # Compute distance from center (frequency radius)
    D = np.sqrt(U**2 + V**2)

    if filter_type == 'low':
        # Ideal Low-Pass Filter: 1 inside radius, 0 outside
        H = (D <= cutoff).astype(float)
    else:  # high-pass
        # Ideal High-Pass Filter: 0 inside radius, 1 outside
        H = (D > cutoff).astype(float)

    return H

# Get image dimensions
M, N = cat_array.shape

# Create filters at THREE different cutoffs for visible effects
cutoff_conservative = min(M, N) // 4   # 25% - subtle blur
cutoff_default = min(M, N) // 8         # 12.5% - moderate effect
cutoff_aggressive = min(M, N) // 16     # 6.25% - strong blur

print(f"Image dimensions: {M} x {N}")
print(f"Conservative cutoff (25%): {cutoff_conservative} pixels - subtle blur")
print(f"Default cutoff (12.5%): {cutoff_default} pixels - moderate effect")
print(f"Aggressive cutoff (6.25%): {cutoff_aggressive} pixels - strong blur")

# Create LPF masks at all cutoffs
lpf_conservative = create_ideal_filter((M, N), cutoff_conservative, 'low')
lpf_default = create_ideal_filter((M, N), cutoff_default, 'low')
lpf_aggressive = create_ideal_filter((M, N), cutoff_aggressive, 'low')

# Create HPF mask (edges look similar at different cutoffs)
hpf_default = create_ideal_filter((M, N), cutoff_default, 'high')

# Visualize all filter masks for comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(lpf_conservative, cmap='gray')
axes[0].set_title(f'LPF Conservative\n(D0={cutoff_conservative}, 25%)')
axes[0].axis('off')

axes[1].imshow(lpf_default, cmap='gray')
axes[1].set_title(f'LPF Default\n(D0={cutoff_default}, 12.5%)')
axes[1].axis('off')

axes[2].imshow(lpf_aggressive, cmap='gray')
axes[2].set_title(f'LPF Aggressive\n(D0={cutoff_aggressive}, 6.25%)')
axes[2].axis('off')

axes[3].imshow(hpf_default, cmap='gray')
axes[3].set_title(f'HPF Default\n(D0={cutoff_default})')
axes[3].axis('off')

plt.tight_layout()
plt.show()

print("\nFilter masks show which frequencies are passed (white = 1) or blocked (black = 0)")

In [ ]:
# Cell 7: Apply Filters with Multiple Cutoff Comparison
# Apply LPF at different cutoffs to cat image
cat_lpf_conservative = np.real(ifft2(fftshift(cat_fft_shift * lpf_conservative)))
cat_lpf_default = np.real(ifft2(fftshift(cat_fft_shift * lpf_default)))
cat_lpf_aggressive = np.real(ifft2(fftshift(cat_fft_shift * lpf_aggressive)))

# Apply HPF to cat image
cat_hpf = np.real(ifft2(fftshift(cat_fft_shift * hpf_default)))

# Apply LPF at default cutoff to dog image
dog_lpf_default = np.real(ifft2(fftshift(dog_fft_shift * lpf_default)))

# Apply HPF to dog image
dog_hpf = np.real(ifft2(fftshift(dog_fft_shift * hpf_default)))

print("Filtering completed successfully!")
print(f"\nCat LPF results:")
print(f"  Conservative (25%): range [{cat_lpf_conservative.min():.2f}, {cat_lpf_conservative.max():.2f}]")
print(f"  Default (12.5%): range [{cat_lpf_default.min():.2f}, {cat_lpf_default.max():.2f}]")
print(f"  Aggressive (6.25%): range [{cat_lpf_aggressive.min():.2f}, {cat_lpf_aggressive.max():.2f}]")
print(f"\nCat HPF result:")
print(f"  Default (12.5%): range [{cat_hpf.min():.2f}, {cat_hpf.max():.2f}]")

In [ ]:
# Cell 8: Display Filtered Results with Clear Cutoff Comparison
# Display original vs filtered images - showing multiple cutoff effects

# Row 1: Cat - Original and LPF at different cutoffs
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

# Top row: Cat images
axes[0, 0].imshow(cat_array, cmap='gray')
axes[0, 0].set_title('Cat - Original')
axes[0, 0].axis('off')

axes[0, 1].imshow(cat_lpf_conservative, cmap='gray')
axes[0, 1].set_title(f'LPF Conservative (25%)\nD0={cutoff_conservative}')
axes[0, 1].axis('off')

axes[0, 2].imshow(cat_lpf_default, cmap='gray')
axes[0, 2].set_title(f'LPF Default (12.5%)\nD0={cutoff_default}')
axes[0, 2].axis('off')

axes[0, 3].imshow(cat_lpf_aggressive, cmap='gray')
axes[0, 3].set_title(f'LPF Aggressive (6.25%)\nD0={cutoff_aggressive}')
axes[0, 3].axis('off')

# Bottom row: Dog images and HPF
axes[1, 0].imshow(dog_array, cmap='gray')
axes[1, 0].set_title('Dog - Original')
axes[1, 0].axis('off')

axes[1, 1].imshow(dog_lpf_default, cmap='gray')
axes[1, 1].set_title(f'LPF Default (12.5%)\nD0={cutoff_default}')
axes[1, 1].axis('off')

axes[1, 2].imshow(dog_hpf, cmap='gray')
axes[1, 2].set_title('HPF - Edge Enhancement')
axes[1, 2].axis('off')

# Side-by-side comparison: Original vs strongest blur
axes[1, 3].imshow(cat_hpf, cmap='gray')
axes[1, 3].set_title('HPF - Shows Edges Only')
axes[1, 3].axis('off')

plt.tight_layout()
plt.show()

print("\n" + "=" * 70)
print("OBSERVATIONS - LPF Effect (Blur):")
print("=" * 70)
print(f"- Conservative (25%): Subtle blur - still shows fine details")
print(f"- Default (12.5%): Moderate blur - fur texture noticeably softened")
print(f"- Aggressive (6.25%): Strong blur - major features visible, fine detail lost")
print("\n" + "=" * 70)
print("OBSERVATIONS - HPF Effect (Edge Enhancement):")
print("=" * 70)
print("- Smooth regions become dark/black (low frequencies removed)")
print("- Edges between areas are highlighted (high frequencies preserved)")
print("- Result looks like a sketch/outline of the original image")

---
## Part B: Audio Signal Processing

In [ ]:
# Cell 9: Load Audio and Display Metadata
# Load the audio file
audio_path = 'song_with_2piccolo.wav'
y, sr = librosa.load(audio_path, sr=None)  # sr=None keeps original sampling rate

# Display audio metadata
duration = len(y) / sr
print(f"Audio file: {audio_path}")
print(f"Sampling rate: {sr} Hz")
print(f"Duration: {duration:.2f} seconds")
print(f"Number of samples: {len(y)}")
print(f"Signal dtype: {y.dtype}")
print(f"Signal range: [{y.min():.4f}, {y.max():.4f}]")

In [ ]:
# Cell 10: Normalize and Display Waveform
# Normalize audio to [-1, 1] range
y_normalized = y / np.max(np.abs(y))

print(f"Normalized range: [{y_normalized.min():.4f}, {y_normalized.max():.4f}]")

# Plot waveform
plt.figure(figsize=(12, 4))
librosa.display.waveshow(y_normalized, sr=sr)
plt.title("Audio Waveform - Song with 2 Piccolo")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nWaveform shows amplitude variation over time.")
print("The piccolo melody creates distinct amplitude patterns.")

In [ ]:
# Cell 11: STFT and Spectrogram
# Compute Short-Time Fourier Transform (STFT)
D = librosa.stft(y)

print(f"STFT shape: {D.shape}")
print(f"STFT dtype: {D.dtype}")

# Convert to dB scale
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

print(f"dB range: [{S_db.min():.2f}, {S_db.max():.2f}] dB")

# Plot spectrogram
plt.figure(figsize=(12, 5))
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz', cmap='magma')
plt.colorbar(format='%+2.0f dB')
plt.title("Spectrogram - Song with 2 Piccolo")
plt.xlabel("Time (s)")
plt.ylabel("Frequency (Hz)")
plt.tight_layout()
plt.show()

print("\nSpectrogram shows frequency content over time.")
print("Brighter colors = higher energy at that frequency.")

In [ ]:
# Cell 12: Spectral Analysis - Identify Dominant Frequencies
# Compute mean spectrum across time
mean_spectrum = np.mean(np.abs(D), axis=1)
frequencies = librosa.fft_frequencies(sr=sr)

# Plot power spectral density
plt.figure(figsize=(12, 4))
plt.semilogy(frequencies, mean_spectrum)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Mean Magnitude')
plt.title('Average Power Spectrum')
plt.grid(True, alpha=0.3)
plt.xlim([0, sr/2])
plt.tight_layout()
plt.show()

# Find peak frequencies (dominant harmonics)
from scipy.signal import find_peaks

peaks, properties = find_peaks(mean_spectrum, height=np.max(mean_spectrum)*0.1)
peak_frequencies = frequencies[peaks]
peak_magnitudes = mean_spectrum[peaks]

# Sort by magnitude and show top 10
sorted_idx = np.argsort(peak_magnitudes)[::-1][:10]
print("\nTop 10 Dominant Frequencies:")
print("-" * 30)
for i, idx in enumerate(sorted_idx):
    print(f"{i+1}. {peak_frequencies[idx]:.1f} Hz (magnitude: {peak_magnitudes[idx]:.4f})")

print("\nPiccolo frequencies typically range from 500 Hz to 4 kHz.")

In [ ]:
# Cell 13: Summary and Conclusions
print("=" * 70)
print("EE200 Signal Processing Project - Summary")
print("=" * 70)

print("\n--- IMAGE PROCESSING ---")
print(f"✓ Loaded grayscale images: cat ({cat_array.shape}), dog ({dog_array.shape})")
print(f"✓ Performed basic operations: resize, crop, rotate")
print(f"✓ Computed 2D DFT and displayed magnitude/phase spectra")
print(f"✓ Designed ideal LPF at 3 cutoffs (6.25%, 12.5%, 25%)")
print(f"✓ Designed ideal HPF for edge enhancement")
print(f"✓ Applied frequency domain filtering - visible blur and edge effects")

print("\n--- AUDIO PROCESSING ---")
print(f"✓ Loaded audio: '{audio_path}'")
print(f"✓ Sampling rate: {sr} Hz, Duration: {duration:.2f}s")
print(f"✓ Plotted normalized waveform")
print(f"✓ Computed STFT and dB spectrogram")
print(f"✓ Identified dominant frequencies in piccolo range (500Hz-4kHz)")

print("\n" + "=" * 70)
print("Project completed successfully!")
print("=" * 70)
print("\nKEY CONCEPTS DEMONSTRATED:")
print("- 2D DFT: Transforms image to frequency domain")
print("- LPF: Removes high frequencies → Blur effect")
print("- HPF: Removes low frequencies → Edge enhancement")
print("- STFT: Time-frequency analysis for audio signals")
print("- Spectrogram: Visual representation of frequency content over time")